# SAMAYA Edge AI: SOTA Emotion Recognition (Google Colab — Optimized)

Notebook ini dioptimalkan khusus untuk **Google Colab T4 GPU**. Fitur utama:
- **Mixed Precision (float16)** — training ~2x lebih cepat
- **EfficientNet-B0** + 3-Phase Gradual Unfreezing
- **Oversampling** untuk menangani ketidakseimbangan kelas
- **Augmentasi yang Diperkaya** — brightness, translation, zoom, rotation
- **Test-Time Augmentation (TTA)** pada evaluasi akhir

## 1. Hubungkan Google Drive
Jalankan sel ini agar dataset dapat diakses dan model tersimpan secara permanen.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Ekstraksi Dataset & Bersihkan Cache
Pastikan dataset `.zip` sudah diunggah ke folder `PKB-Emotion-Recognition` di Google Drive Anda.

In [ ]:
import os
import shutil

drive_project_dir = '/content/drive/MyDrive/PKB-Emotion-Recognition'
os.makedirs(drive_project_dir, exist_ok=True)

if not os.path.exists('/content/dataset_rafdb'):
    print("Mengekstrak RAF-DB...")
    !unzip -q "/content/drive/MyDrive/PKB-Emotion-Recognition/dataset_rafdb.zip" -d "/content/dataset_rafdb"
    print("RAF-DB selesai diekstrak!")
else:
    print("Dataset RAF-DB sudah siap.")

if not os.path.exists('/content/dataset_fer2013'):
    print("Mengekstrak FER-2013...")
    !unzip -q "/content/drive/MyDrive/PKB-Emotion-Recognition/dataset_fer2013.zip" -d "/content/dataset_fer2013"
    print("FER-2013 selesai diekstrak!")
else:
    print("Dataset FER-2013 sudah siap.")

if os.path.exists('/content/tf_cache'):
    shutil.rmtree('/content/tf_cache')
    print("Cache lama dihapus.")

## 3. Cek GPU & Aktifkan Mixed Precision
Pastikan runtime diubah ke **T4 GPU** (Runtime → Change runtime type → T4 GPU).

In [ ]:
import tensorflow as tf
import numpy as np
import random
import gc

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU Terdeteksi: {gpus}")
    except RuntimeError as e:
        print(e)
else:
    print("PERINGATAN: GPU tidak ditemukan! Ubah runtime ke T4 GPU.")

tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f"[INFO] Mixed Precision: {tf.keras.mixed_precision.global_policy().name}")

## 4. Konfigurasi & Muat Pipa Data (Oversampling & SSD Caching)

In [ ]:
import hashlib
import matplotlib.pyplot as plt
from collections import Counter

IMG_SIZE = 224
BATCH_SIZE = 64
NUM_CLASSES = 7

FER_TRAIN_DIR = '/content/dataset_fer2013/train'
FER_TEST_DIR = '/content/dataset_fer2013/test'
RAF_TRAIN_DIR = '/content/dataset_rafdb/train'
RAF_TEST_DIR = '/content/dataset_rafdb/test'

def load_and_preprocess_image(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return img, label

def get_balanced_dataset(data_dir, is_training=True):
    class_names = sorted([d for d in os.listdir(data_dir)
                          if os.path.isdir(os.path.join(data_dir, d))])
    assert len(class_names) == NUM_CLASSES, \
        f"Expected {NUM_CLASSES} classes, got {len(class_names)}: {class_names}"

    paths, labels = [], []
    for i, c in enumerate(class_names):
        class_dir = os.path.join(data_dir, c)
        class_paths = [os.path.join(class_dir, f) for f in os.listdir(class_dir)
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        paths.extend(class_paths)
        labels.extend([i] * len(class_paths))
        print(f"  Kelas {c}: {len(class_paths)} sampel")

    print(f"\n[INFO] {data_dir} — Total: {len(paths)} sampel")

    if is_training:
        counter = Counter(labels)
        max_count = max(counter.values())
        balanced_paths, balanced_labels = [], []
        for i in range(NUM_CLASSES):
            class_indices = [idx for idx, label in enumerate(labels) if label == i]
            duplicated = np.random.choice(class_indices, size=max_count, replace=True)
            balanced_paths.extend([paths[idx] for idx in duplicated])
            balanced_labels.extend([labels[idx] for idx in duplicated])
        print(f"[INFO] Setelah Oversampling: {len(balanced_paths)} sampel (seimbang)")
        final_paths, final_labels = balanced_paths, balanced_labels
    else:
        final_paths, final_labels = paths, labels

    final_labels_cat = tf.keras.utils.to_categorical(final_labels, num_classes=NUM_CLASSES)
    ds = tf.data.Dataset.from_tensor_slices((final_paths, final_labels_cat))
    ds = ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

    cache_dir = "/content/tf_cache"
    os.makedirs(cache_dir, exist_ok=True)
    path_hash = hashlib.md5(data_dir.encode('utf-8')).hexdigest()
    tag = "train" if is_training else "val"
    ds = ds.cache(os.path.join(cache_dir, f"cache_{path_hash}_{tag}"))

    if is_training:
        ds = ds.shuffle(buffer_size=2048)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

raf_train_ds = get_balanced_dataset(RAF_TRAIN_DIR, is_training=True)
raf_val_ds = get_balanced_dataset(RAF_TEST_DIR, is_training=False)

## 5. Pendefinisian Model (EfficientNet-B0 + Augmentasi Diperkaya)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import (
    Dense, GlobalAveragePooling2D, Dropout, Input,
    RandomFlip, RandomRotation, RandomContrast, RandomZoom,
    RandomBrightness, RandomTranslation
)
from tensorflow.keras.models import Model, Sequential

data_augmentation = Sequential([
    RandomFlip("horizontal"),
    RandomRotation(factor=0.10),
    RandomZoom(height_factor=0.10, width_factor=0.10),
    RandomContrast(factor=0.15),
    RandomBrightness(factor=0.10),
    RandomTranslation(height_factor=0.05, width_factor=0.05),
], name="data_augmentation")

def build_model(pretrained_weights=None, trainable_backbone=False):
    inputs = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)

    base_model = EfficientNetB0(weights='imagenet', include_top=False,
                                 input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base_model.trainable = trainable_backbone

    x = base_model(x)
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.4)(x)
    predictions = Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = Model(inputs=inputs, outputs=predictions)

    # Muat bobot pretrained jika tersedia
    if pretrained_weights and os.path.exists(pretrained_weights):
        print(f"[INFO] Memuat bobot dari: {pretrained_weights}")
        try:
            model.load_weights(pretrained_weights)
            print("[INFO] Semua bobot berhasil dimuat!")
        except (ValueError, Exception) as e:
            print(f"[WARN] Gagal load penuh ({type(e).__name__}), mencoba backbone saja...")
            try:
                old_model = tf.keras.models.load_model(pretrained_weights, compile=False)
                for layer in old_model.layers:
                    if 'efficientnet' in layer.name.lower():
                        base_model.set_weights(layer.get_weights())
                        print("[INFO] Bobot backbone EfficientNet berhasil dimuat!")
                        break
                del old_model
                gc.collect()
            except Exception as e2:
                print(f"[WARN] Gagal memuat backbone: {e2}. Menggunakan bobot ImageNet.")

    return model, base_model

## 6. Pre-training pada FER-2013 (Opsional)

> **Rekomendasi: `RUN_FER_PRETRAINING = False`** (default).
> Transfer learning langsung dari ImageNet ke RAF-DB secara empiris lebih stabil dan akurat dibanding melalui FER-2013 (kualitas gambar rendah, label berisik).

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.losses import CategoricalFocalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay, CosineDecayRestarts

RUN_FER_PRETRAINING = False
fer_weights_path = '/content/drive/MyDrive/PKB-Emotion-Recognition/fer2013_efficientnetb0.keras'

if RUN_FER_PRETRAINING:
    print("=== TAHAP 1: PRE-TRAINING PADA FER-2013 ===")
    fer_train_ds = get_balanced_dataset(FER_TRAIN_DIR, is_training=True)
    fer_val_ds = get_balanced_dataset(FER_TEST_DIR, is_training=False)

    fer_model, _ = build_model(trainable_backbone=True)
    fer_steps = len(fer_train_ds)
    lr_fer = CosineDecayRestarts(initial_learning_rate=1e-3, first_decay_steps=fer_steps * 5)

    fer_model.compile(
        optimizer=Adam(learning_rate=lr_fer),
        loss=CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0, label_smoothing=0.1),
        metrics=['accuracy']
    )

    fer_callbacks = [
        ModelCheckpoint(fer_weights_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
    ]

    fer_model.fit(fer_train_ds, validation_data=fer_val_ds, epochs=10, callbacks=fer_callbacks)
    print("[SUCCESS] Pre-training FER-2013 Selesai.")
    del fer_model
    gc.collect()
else:
    print("[INFO] Pre-training FER-2013 dilewati. Langsung dari ImageNet ke RAF-DB.")

## 7. Fine-Tuning RAF-DB — Fase 2.1 (Train Head Only)
Backbone EfficientNet-B0 dibekukan. Hanya classification head yang dilatih.

In [ ]:
print("=== FASE 2.1: Train Head Only ===")

# Hanya muat bobot FER jika pre-training dijalankan di sesi ini
weights_to_load = fer_weights_path if (RUN_FER_PRETRAINING and os.path.exists(fer_weights_path)) else None
raf_model, raf_base = build_model(pretrained_weights=weights_to_load, trainable_backbone=False)

steps_per_epoch = len(raf_train_ds)
lr_p1 = CosineDecayRestarts(initial_learning_rate=1e-3, first_decay_steps=steps_per_epoch * 3)
raf_model.compile(
    optimizer=Adam(learning_rate=lr_p1),
    loss=CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0, label_smoothing=0.1),
    metrics=['accuracy']
)

best_model_path = '/content/drive/MyDrive/PKB-Emotion-Recognition/samaya_rafdb_sota.keras'

callbacks_p1 = [
    ModelCheckpoint(best_model_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
]

history_p1 = raf_model.fit(raf_train_ds, validation_data=raf_val_ds, epochs=10, callbacks=callbacks_p1)

## 8. Fine-Tuning RAF-DB — Fase 2.2 (Top 30 Layers)
30 layer teratas backbone di-unfreeze dengan learning rate rendah.

In [ ]:
print("=== FASE 2.2: Fine-Tune Top 30 Layers ===")

raf_base.trainable = True
for layer in raf_base.layers[:-30]:
    layer.trainable = False

EPOCHS_P2 = 15
lr_p2 = CosineDecay(initial_learning_rate=1e-4, decay_steps=steps_per_epoch * EPOCHS_P2)
raf_model.compile(
    optimizer=Adam(learning_rate=lr_p2),
    loss=CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0, label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_p2 = [
    ModelCheckpoint(best_model_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
]

history_p2 = raf_model.fit(raf_train_ds, validation_data=raf_val_ds, epochs=EPOCHS_P2, callbacks=callbacks_p2)

## 9. Fine-Tuning RAF-DB — Fase 2.3 (Unfreeze All)
Seluruh model di-unfreeze dengan learning rate sangat kecil.

In [ ]:
print("=== FASE 2.3: Unfreeze All ===")

raf_base.trainable = True

EPOCHS_P3 = 10
lr_p3 = CosineDecay(initial_learning_rate=1e-5, decay_steps=steps_per_epoch * EPOCHS_P3)
raf_model.compile(
    optimizer=Adam(learning_rate=lr_p3),
    loss=CategoricalFocalCrossentropy(alpha=0.25, gamma=2.0, label_smoothing=0.1),
    metrics=['accuracy']
)

callbacks_p3 = [
    ModelCheckpoint(best_model_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1)
]

history_p3 = raf_model.fit(raf_train_ds, validation_data=raf_val_ds, epochs=EPOCHS_P3, callbacks=callbacks_p3)

## 10. Evaluasi Akhir dengan Test-Time Augmentation (TTA)

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.models import load_model

print("=== EVALUASI AKHIR ===")
tf.keras.mixed_precision.set_global_policy('mixed_float16')

best_model = load_model(best_model_path,
    custom_objects={'CategoricalFocalCrossentropy': CategoricalFocalCrossentropy})

print("Prediksi normal...")
predictions_normal = best_model.predict(raf_val_ds, verbose=1)

print("Prediksi dengan horizontal flip (TTA)...")
def flip_images(images, labels):
    return tf.image.flip_left_right(images), labels

raf_val_flipped = raf_val_ds.map(flip_images)
predictions_flipped = best_model.predict(raf_val_flipped, verbose=1)

predictions = (predictions_normal + predictions_flipped) / 2.0
y_pred = np.argmax(predictions, axis=1)

y_true = np.concatenate([y for x, y in raf_val_ds], axis=0)
y_true = np.argmax(y_true, axis=1)

emotion_names = ['surprise', 'fear', 'disgust', 'happy', 'sad', 'angry', 'neutral']
print("\n[INFO] Label Mapping (folder -> emosi):")
for i, name in enumerate(emotion_names):
    print(f"  Folder {i+1} -> {name}")

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_true, y_pred, target_names=emotion_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=emotion_names, yticklabels=emotion_names)
plt.title('Confusion Matrix — SAMAYA EfficientNet-B0 SOTA')
plt.ylabel('Label Asli (True)')
plt.xlabel('Prediksi Model (Predicted)')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/PKB-Emotion-Recognition/samaya_rafdb_confusion_matrix_sota.png', dpi=300)
plt.show()

accuracy = accuracy_score(y_true, y_pred)
print(f"\nAKURASI AKHIR (dengan TTA): {accuracy:.4f} ({accuracy*100:.2f}%)")